# Phase 9 — Explanation and price recommendation

This notebook explains the saved tuned CatBoost model and converts each point estimate into a calibrated 80% price range with a support-aware confidence label.

**Hard guardrail:** the model is not retrained here, and the 2,287 test rows are not transformed, explained, predicted, or evaluated.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

summary = json.loads((PROJECT_ROOT / 'reports/metrics/price_recommendation_summary.json').read_text(encoding='utf-8'))
print(summary['phase'])
print('Model:', summary['model']['name'])
print('Model retrained in Phase 9:', summary['model']['retrained_in_phase9'])
print('Test rows used:', summary['split']['test_rows_used'])

## 1. Two kinds of explanation

CatBoost's native `PredictionValuesChange` importance summarizes how much each feature changes predictions across the model. TreeSHAP provides a second, additive explanation: each feature receives a positive or negative contribution for every individual car.

The model predicts `log1p(price)`, so SHAP contributions are additive on the log-price scale. Their effects become approximately multiplicative after converting back to rupees.

In [ ]:
importance = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase9_global_feature_importance.csv')
importance[['feature', 'catboost_prediction_values_change_importance', 'mean_absolute_shap_log_contribution', 'shap_importance_rank']]

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/25_global_feature_importance.png')))
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/26_shap_summary.png')))

The strongest global TreeSHAP features are vehicle age, maximum power, engine size, model, and transmission. Red points raise an individual prediction and blue points lower it. Color shows contribution direction—not whether a raw feature value is inherently good or bad.

SHAP explains the fitted model's associations. It does **not** prove that changing a feature would causally change the selling price.

## 2. Why the range is not ±10%

A fixed percentage would be arbitrary. Instead, the fixed validation rows are divided into a calibration half and a separate interval-evaluation half. Absolute log-price residuals from calibration determine the lower and upper multipliers.

Calibration is grouped by the model's **predicted** price band. That matters because a new car has no known actual price; using its actual band would leak the answer.

In [ ]:
calibration = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase9_interval_calibration.csv')
calibration

## 3. Did the 80% range work?

Coverage is measured only on the 1,144 interval-evaluation rows that did not set the residual quantiles. About 83.8% of their actual prices fall inside the calibrated range, which is reasonably close to—and slightly more conservative than—the nominal 80%.

In [ ]:
coverage = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase9_interval_coverage.csv')
coverage

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/28_interval_coverage.png')))

The ₹5–10 lakh band is almost exactly at 80%. The two expensive bands are conservative, but they also have much smaller evaluation samples. This is a development-stage interval check because validation supported earlier model development; the sealed test set is still needed for final confirmation later.

## 4. Confidence logic

- **High:** at least 50 matching training brand-model rows, all numeric inputs inside their common 1st–99th percentile ranges, and a narrow calibrated interval.
- **Medium:** adequate support, but an unusual input or moderate interval width.
- **Low:** unseen or fewer than 10 matching brand-model rows, a numeric input outside the observed training range, or a very wide calibrated interval.

Low confidence is a real warning: its empirical evaluation coverage is 74.6%, below the nominal 80%.

In [ ]:
recommendations = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase9_validation_recommendations.csv')
evaluation = recommendations[recommendations['interval_role'].eq('interval_evaluation')]
evaluation['confidence'].value_counts().rename_axis('confidence').to_frame('rows')

## 5. Four individual explanations

In [ ]:
display(Image(filename=str(PROJECT_ROOT / 'reports/figures/27_individual_shap_explanations.png')))
explanations = pd.read_csv(PROJECT_ROOT / 'reports/tables/phase9_individual_explanations.csv')
for case, group in explanations.groupby('case', sort=False):
    print(f'\n{case}')
    print(group.iloc[0]['case_summary'])

## 6. Use the reusable recommendation function

`recommend_price()` accepts the same 11 features used by CatBoost and returns the point estimate, calibrated range, confidence reason, and five strongest TreeSHAP drivers.

In [ ]:
from src.explain_recommend import recommend_price
from src.features import BASE_FEATURE_COLUMNS

example_car = recommendations.loc[0, list(BASE_FEATURE_COLUMNS)].to_dict()
result = recommend_price(example_car)
pd.Series({
    'recommended_price': result['recommended_price'],
    'range_lower': result['range_lower'],
    'range_upper': result['range_upper'],
    'confidence': result['confidence'],
    'confidence_reason': result['confidence_reason'],
})

In [ ]:
pd.DataFrame(result['top_shap_drivers'])[['feature', 'value', 'direction', 'approximate_price_signal_percentage', 'explanation']]

## Phase 9 conclusion

Tuned CatBoost now provides four connected outputs: a price recommendation, an 80% calibrated range, a support-aware confidence label, and local TreeSHAP drivers. It remains the Phase 10 candidate rather than a finally confirmed production model.

## Reproducing Phase 9

From the project root, run:

```bash
python -m src.explain_recommend
```